In [0]:
from pyspark.sql.functions import current_timestamp, col, lit, to_date, trim, lower, upper, when

In [0]:
%run "../includes/common_functions"

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS f1.silver;

In [0]:
circuits_df = spark.read\
    .table("f1.bronze.circuits")

In [0]:

circuits_renamed_df = circuits_df\
    .withColumnRenamed("name", "circuit_name")\
    .withColumnRenamed("circuitId","circuit_id")\
    .withColumnRenamed("circuitRef","circuit_ref")\
    .withColumnRenamed("lat","latitude")\
    .withColumnRenamed("lng","longitude")\
    .withColumnRenamed("alt", "altitude")

In [0]:
circuits_renamed_df = circuits_renamed_df\
    .withColumn("circuit_ref", trim(col("circuit_ref")))\
    .withColumn("location", trim(col("location")))\
    .withColumn("country", trim(col("country")))\
    .withColumn("url", trim(col("url")))

In [0]:
circuits_silver_df = circuits_renamed_df.dropDuplicates(["circuit_id"])

In [0]:
circuits_silver_df = circuits_silver_df.filter(
    col("circuit_id").isNotNull()
)

In [0]:
circuits_final_df = add_ingestion_date(circuits_silver_df) \
    .withColumn("environment", lit("production")) \
    .withColumn("file_date", to_date(lit("2026-05-07")))

In [0]:
circuits_final_clean_df = circuits_final_df\
    .withColumn("country", when(trim(col("country")) == 'USA', "United States")\
        .when(trim(col("country")) == "UK", "United Kingdom")\
        .when(trim(col("country")) == "UAE", "United Arab Emirates")\
        .otherwise(trim(col("country")))
    )
    

In [0]:
circuits_final_clean_df.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("f1.silver.circuits")

In [0]:
%sql
SELECT *
FROM f1.silver.circuits
LIMIT 10;